[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/08-bulk_entity_resolution_for_data_agents.ipynb)

In [1]:
# !pip install mbox openai python-dotenv

# Bulk Entity Resolution for Data Agents

Every notebook so far resolved one query at a time. A data-cleaning agent reconciling an incoming batch against a master table has a different shape of problem: dozens, hundreds, or thousands of rows, most of which are an obviously correct match with some noise, a few of which are genuinely new, and a small remainder that actually need judgment. Running an LLM over every single row to decide "is this a match" is slow and, as this notebook shows, mostly unnecessary. M|BOX's batch matching resolves the confident majority deterministically in one call, so the only rows that ever reach an LLM, or a human, are the ones that actually need one.

In this notebook you will:

1. Match an entire incoming batch against a master table in a single call, not a loop
2. Split the results into buckets by confidence, automatically, using vectorized pandas, not a per-row loop
3. Learn to tell genuine ambiguity apart from a fixable data-quality pattern, they are not the same problem and don't belong in the same bucket
4. Send only the one row that actually needs it to an LLM for adjudication

> Note: this notebook makes one real call to the OpenAI API, for the single row that genuinely needs judgment. To run it, put an `OPENAI_API_KEY` in a `.env` file in this directory.

In [2]:
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

## 1. The master table and the incoming batch

`crm_master.csv` is 31 existing company records. `import_batch.csv` is 30 incoming rows from some other system, company names with the usual real-world noise: typos, transposed letters, inconsistent capitalization, and legal suffixes (`Inc`, `LLC`, `Corp`) added, dropped, or occasionally duplicated.

In [3]:
master = pd.read_csv("datasets/crm_master.csv")
batch = pd.read_csv("datasets/import_batch.csv")
print(f"{len(master)} master records, {len(batch)} incoming rows")
batch.head()

31 master records, 30 incoming rows


,company_name,contact_domain
0,Willow Creek Bakery,willowcreekb.com
1,Granite Peak Legal Corp Corp,granitepeakl.com
2,Cascade Analytics Ltd Corp,cascadeanaly.com
3,Nortwhind Logistics LLC,northwindlog.com
4,Harbovriew Realty Co,harborviewre.com


## 2. One match call for the whole batch

`index.match(queries=batch)` runs every row in `batch` against the compiled index in a single call, `max_results=2` keeps the top two candidates per row so a genuine tie is visible, not just the winner.

In [4]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

index = TableIndexer.create_index(master, index_columns=["company_name"], tmp_dir="tmp_index_crm")
config = TableRecallConfig(
    fields=[TableRecallFieldConfig(input_column="company_name", indexed_column="company_name",
                                    minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
    max_results=2, min_total_match_value=0, include_field_scores=True
)
matches = index.match(queries=batch[["company_name"]], config=config)
print(f"{len(matches)} result rows for {len(batch)} queries, because a handful returned two close candidates")

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


32 result rows for 30 queries, because a handful returned two close candidates


## 3. Bucket the whole result set at once

`query_row` ties each result row back to its original query. Grouping by it gives, for every incoming row, its best score, its second-best score if one exists, and how far apart they are, all without a Python loop over 30 rows.

In [5]:
g = matches.groupby("query_row")
summary = g.agg(top_score=("company_name_score", "first"),
                 top_candidate=("company_name_candidate", "first"),
                 n_candidates=("company_name_candidate", "size"))

matches["rank"] = matches.groupby("query_row").cumcount()
runner_up = matches[matches["rank"] == 1].set_index("query_row")["company_name_score"]
summary["second_score"] = summary.index.map(runner_up).fillna(0).astype(int)
summary["gap"] = summary["top_score"] - summary["second_score"]
summary["query_name"] = summary.index.map(batch["company_name"])

HIGH, AMBIGUOUS_GAP = 85, 5

def bucket(row):
    if row["top_score"] == 0:
        return "NO_MATCH"
    if row["top_score"] < HIGH:
        return "NEEDS_REVIEW"
    if row["n_candidates"] > 1 and row["gap"] < AMBIGUOUS_GAP and row["top_score"] < 95:
        return "NEEDS_REVIEW"
    return "AUTO_MERGE"

summary["bucket"] = summary.apply(bucket, axis=1)
summary["bucket"].value_counts()

bucket
AUTO_MERGE      21
NEEDS_REVIEW     6
NO_MATCH         3
Name: count, dtype: int64

21 of 30 rows are confident enough to merge automatically, no LLM call involved, no human involved. 3 have no plausible match at all, they're genuinely new companies. That leaves 6 rows worth a closer look, and it would be a mistake to treat all 6 the same way.

## 4. Two very different reasons a row lands in `NEEDS_REVIEW`

Split the 6 by whether they actually have a competing second candidate, or just one plausible candidate that scored lower than the confidence bar.

In [6]:
review = summary[summary["bucket"] == "NEEDS_REVIEW"]
ambiguous = review[review["n_candidates"] > 1]
low_confidence = review[review["n_candidates"] == 1]

print("Genuinely ambiguous, a real second candidate close behind the first:")
print(ambiguous[["query_name", "top_candidate", "top_score", "second_score"]].to_string(), "\n")

print("Only one plausible candidate, just scored below the confidence bar:")
print(low_confidence[["query_name", "top_candidate", "top_score"]].to_string())

Genuinely ambiguous, a real second candidate close behind the first:
                           query_name           top_candidate  top_score  second_score
query_row                                                                             
8          Ironwood Manufacturing LLC  Ironwood Manufacturing         68            64 

Only one plausible candidate, just scored below the confidence bar:
                             query_name            top_candidate  top_score
query_row                                                                  
1          Granite Peak Legal Corp Corp  Granite Peak Legal Corp         57
2            Cascade Analytics Ltd Corp    Cascade Analytics Ltd         55
10                 Redstone Capital LLC         Redstone Capital         57
15              Coral Bay Insurance Inc      Coral Bay Insurance         61
19                Silver Fox Media Corp         Silver Fox Media         43


Those five low-confidence rows aren't ambiguous, `Silver Fox Media Corp` has exactly one reasonable candidate, `Silver Fox Media`, mbox just isn't confident enough to say so at a `43`. Look at what they have in common: every one of them differs from its correct match by a legal suffix, `Corp`, `LLC`, `Inc`, added, dropped, or doubled up. That's not five unrelated judgment calls, it's the same fixable data-quality pattern five times. The right fix is normalizing suffixes before matching, the kind of thing `02-data-harmonization/` covers, not spending an LLM call on each one, and not lowering the confidence bar either, which would just as happily wave through a genuinely wrong match somewhere else in the batch.

The one row in `ambiguous`, by contrast, is a real fork: `Ironwood Manufacturing LLC` is a close match to *two different real companies*, `Ironwood Manufacturing` and `Ironwood Manufacturing West`. No amount of suffix normalization resolves that, it needs a judgment call, and this is the one row actually worth sending somewhere for one.

## 5. Send only the genuinely ambiguous row to an LLM

One row, both candidates, and whatever other context the incoming record carries, here the contact domain, which the raw name-matching score never even looked at.

In [7]:
import json
from openai import OpenAI
client = OpenAI()

row = ambiguous.iloc[0]
incoming = batch.loc[row.name]
candidates = master[master["company_name"].str.contains("Ironwood")]

tools = [{
    "type": "function",
    "function": {
        "name": "pick_match",
        "description": "Pick which candidate company record the incoming record actually refers to.",
        "parameters": {
            "type": "object",
            "properties": {"chosen_org_id": {"type": "string"}, "reason": {"type": "string"}},
            "required": ["chosen_org_id", "reason"]
        }
    }
}]

candidate_lines = "\n".join(f"- {r.org_id}: '{r.company_name}', contact_email='{r.contact_email}'" for r in candidates.itertuples())
prompt = (
    f"An incoming record needs to be matched to exactly one existing company.\n\n"
    f"Incoming: company_name='{incoming['company_name']}', contact_domain='{incoming['contact_domain']}'\n\n"
    f"Candidates:\n{candidate_lines}\n\n"
    "Pick the one candidate this incoming record actually refers to."
)
response = client.chat.completions.create(
    model="gpt-4o", messages=[{"role": "user", "content": prompt}],
    tools=tools, tool_choice={"type": "function", "function": {"name": "pick_match"}}
)
decision = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
decision

{'chosen_org_id': 'ORG-1004',
 'reason': "The incoming record 'company_name=Ironwood Manufacturing LLC' is most similar to 'Ironwood Manufacturing' from ORG-1004. Additionally, the contact domain 'ironwoodmanu.com' closely matches the contact email domain 'contact@ironwoodmanu.com' in ORG-1004."}

The matching email domain settled it, `ironwoodmanu.com` belongs to the base company, not `Ironwood Manufacturing West`. That's a fact M|BOX's name-only match never had access to, and exactly the kind of extra context an LLM call is worth spending on, for the one row that actually needed it.

## 6. The tally

- **21 rows** merged automatically, zero LLM calls
- **3 rows** flagged as genuinely new records, zero LLM calls
- **5 rows** flagged for a one-time suffix-normalization fix upstream, zero LLM calls, and zero wasted on treating a systematic pattern as five separate judgment calls
- **1 row** sent to an LLM, because it was the only one that actually needed a judgment call

## Practical notes

**Don't route everything below a confidence threshold to an LLM.** A low score has more than one cause, genuine ambiguity between real candidates, and a systematic data-quality pattern, look identical at the level of "the score was low." Splitting by whether a real second candidate exists, the way this notebook did, is a cheap, mechanical way to tell them apart before spending anything expensive on either.

**A recurring low-confidence pattern is a signal to fix the pipeline, not a queue to work through.** Five rows failing on the same legal-suffix issue is not five problems, it's one problem with five symptoms. Fixing it once upstream, in harmonization, benefits every future batch, working through the symptoms one by one does not.

**Give the LLM the context M|BOX's field didn't have.** The name-only match couldn't see the contact domain. The one row that actually reached the model got the domain, both candidates' full records, and a specific question, not just "which of these is right."